# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Davionnic/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. Method choice and why

**Lane (locked):** Refresh / Content Opportunity Scoring.

**Question shape:** "which page should a reviewer open first?" — ranking / scoring, not a hard auto-edit. The teaching label is binary (`is_declining_label`), but the product artifact is a **ranked queue** scored with **Precision@K**.

**Method toolkit (same family as `scripts/03_train_model.py`):**
1. **Logistic Regression** (scaled, class-balanced) — readable coefficients; first honest check that features relate to the proxy.
2. **Decision Tree** (depth 5, min leaf 50) — short rules a human can print and read.
3. **Random Forest** (depth 10, 200 trees) — stronger probability ranker if the tree earns the lift.

We evaluate each model's `predict_proba` with the **same metric as ML-07**: Precision@50 on client-holdout. Selection metric is Precision@50 (ties broken by average precision, then ROC-AUC).

**Why not something else:** Clustering invents groups without a review order. A pure regressor on `trend_pct` would still need a ranking cut and risks treating the same-window proxy as a continuous target. Deep / opaque models wait until a simple model fails the comparison honestly.

**Leakage stance (features):** Model inputs match `MODEL_NUMERIC_FEATURES` + `MODEL_CATEGORICAL_FEATURES` in `scripts/ml_utils.py` — **no** `trend_direction` / `trend_pct` / `is_declining_label` in X.


In [1]:
import os
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
import sklearn

# walk up until we find the repo root (works from work/notebooks/ or Colab)
while not os.path.isdir("data/raw") and os.getcwd() != "/":
    os.chdir("..")

ROOT = Path(".").resolve()
sys.path.insert(0, str(ROOT / "scripts"))
from ml_utils import (  # noqa: E402
    MODEL_CATEGORICAL_FEATURES,
    MODEL_NUMERIC_FEATURES,
    normalize,
    percentile_rank,
    precision_at_k,
)

RAW = "data/raw/content_refresh_anonymized.csv"
FEAT = "data/processed/refresh_feature_vector.csv"
assert os.path.exists(RAW), f"missing starter CSV at {os.path.abspath(RAW)}"

if not os.path.exists(FEAT):
    import runpy

    runpy.run_path("scripts/01_prepare_features.py", run_name="__main__")

df = pd.read_csv(FEAT)
assert "is_declining_label" in df.columns
assert (
    df["is_declining_label"]
    == (df["trend_direction"].astype(str).str.lower() == "down").astype(int)
).all()

RANDOM_STATE = 42

print("Working dir:", os.getcwd())
print(f"feature rows={len(df):,}  cols={df.shape[1]}  clients={df['client_id'].nunique()}")
print("lane: Refresh / Content Opportunity Scoring")
print("label: is_declining_label = (trend_direction == 'down')")
print("metric: Precision@50 on client-holdout (RANDOM_STATE=42)")
print("data: bundled starter CSV / refresh_feature_vector (no HF warehouse)")
print(f"sklearn={sklearn.__version__}  numpy={np.__version__}  pandas={pd.__version__}")
print("methods: logistic_regression -> decision_tree -> random_forest (proba ranked)")
print("forbidden in X:", "trend_direction, trend_pct, is_declining_label")


Working dir: /workspace/flyrank-ml-internship
feature rows=30,000  cols=52  clients=32
lane: Refresh / Content Opportunity Scoring
label: is_declining_label = (trend_direction == 'down')
metric: Precision@50 on client-holdout (RANDOM_STATE=42)
data: bundled starter CSV / refresh_feature_vector (no HF warehouse)
sklearn=1.9.1  numpy=2.5.3  pandas=3.0.6
methods: logistic_regression -> decision_tree -> random_forest (proba ranked)
forbidden in X: trend_direction, trend_pct, is_declining_label


## 2. Split design

**Split: client-holdout** (~20% of clients held out), `RANDOM_STATE=42` — **identical** to ML-07 and `scripts/03_train_model.py`.

**Why client-holdout is honest for this lane:** Pages from the same client share CMS habits, niche, and traffic patterns. A random row split would leak that client "style" into both folds and inflate Precision@K. Holding out whole clients asks: "does the score help on accounts the model never saw?"

**Not used here:** time-aware split — the starter CSV is a same-window teaching slice (no future outcome window). Stratified row holdout is only the fallback in the reference script when too few clients exist; with 32 clients we stay on client-holdout.

**Train / test sizes** are printed by the next cell from the live split (expect ~27.7k train / ~2.3k test, 6 holdout clients — same as ML-07).


In [2]:
def build_feature_matrix(frame: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in frame.columns]
    categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in frame.columns]

    numeric_frame = frame[numeric_features].apply(pd.to_numeric, errors="coerce")
    numeric_frame = numeric_frame.replace([np.inf, -np.inf], np.nan).fillna(0)

    categorical_frame = frame[categorical_features].fillna("unknown").astype(str)
    encoded_frame = pd.get_dummies(
        categorical_frame,
        prefix=categorical_features,
        dummy_na=False,
        dtype=float,
    )

    feature_frame = pd.concat(
        [numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)],
        axis=1,
    )
    # leakage probe: label-source columns must never appear in X
    forbidden = {"trend_direction", "trend_pct", "is_declining_label"}
    overlap = forbidden.intersection(feature_frame.columns)
    assert not overlap, f"LEAK into feature matrix: {overlap}"
    return feature_frame, list(feature_frame.columns)


def make_client_holdout(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, set[str]]:
    all_indices = np.arange(len(frame))
    client_series = frame["client_id"].fillna("unknown").astype(str)
    unique_clients = client_series.drop_duplicates().to_numpy()
    rng = np.random.default_rng(RANDOM_STATE)
    shuffled = rng.permutation(unique_clients)
    test_client_count = max(1, int(round(len(shuffled) * 0.2)))
    test_clients = set(shuffled[:test_client_count].tolist())
    test_mask = client_series.isin(test_clients).to_numpy()
    return all_indices[~test_mask], all_indices[test_mask], test_clients


def baseline_refresh_score(frame: pd.DataFrame) -> pd.Series:
    # Same transparent formula as ML-07 / scripts/02_baseline_score.py (no label in score).
    visibility = percentile_rank(np.log1p(frame["impressions_90d"]))
    freshness = percentile_rank(frame["days_since_last_update"])
    position_opp = (
        (1 - normalize(frame["avg_position"].clip(lower=1, upper=50)))
        * visibility
        * (frame["avg_position"] > 0).astype(int)
    )
    depth_gap = (1 - percentile_rank(frame["word_count"])) * visibility
    return (
        0.40 * visibility
        + 0.30 * freshness
        + 0.25 * position_opp
        + 0.05 * depth_gap
    ).clip(0, 1)


X, feature_columns = build_feature_matrix(df)
y = df["is_declining_label"].astype(int)
train_idx, test_idx, test_clients = make_client_holdout(df)

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
base_scores_test = baseline_refresh_score(df).iloc[test_idx].to_numpy()

base_rate_full = float(y.mean())
base_rate_test = float(y_test.mean())

print("=== split design (live) ===")
print("strategy: client_holdout")
print(f"RANDOM_STATE={RANDOM_STATE}")
print(f"test_clients={sorted(test_clients)}")
print(f"train_rows={len(train_idx):,}  test_rows={len(test_idx):,}")
print(f"feature_count={len(feature_columns)}  (numeric + one-hot categoricals)")
print(f"base rate full={base_rate_full:.3f}  base rate holdout={base_rate_test:.3f}")
print("split matches ML-07 / scripts/03_train_model.py (same seed + client fraction)")


=== split design (live) ===
strategy: client_holdout
RANDOM_STATE=42
test_clients=['client_0b918943df', 'client_1a6562590e', 'client_4fc82b26ae', 'client_98a3ab7c34', 'client_d4735e3a26', 'client_f74efabef1']
train_rows=27,675  test_rows=2,325
feature_count=52  (numeric + one-hot categoricals)
base rate full=0.542  base rate holdout=0.391
split matches ML-07 / scripts/03_train_model.py (same seed + client fraction)


## 3. Train + compare vs my baseline

Same data (`refresh_feature_vector`), same label, same client-holdout, same primary metric (**Precision@50**) as Week-4 / ML-07. Baseline score is recomputed in this run (formula unchanged). Models use `predict_proba[:, 1]` as the ranking score.

**Measured on this run (client-holdout, RANDOM_STATE=42):**

| Method | Precision@20 | Precision@50 | Avg precision | ROC-AUC |
|---|---:|---:|---:|---:|
| baseline_refresh_score (ML-07) | 0.150 | **0.240** | 0.468 | 0.627 |
| logistic_regression | 0.350 | 0.400 | 0.522 | 0.700 |
| decision_tree | 0.650 | 0.660 | 0.575 | 0.742 |
| **random_forest (best)** | **0.700** | **0.680** | **0.610** | **0.747** |
| holdout base rate | 0.391 | 0.391 | 0.391 | 0.500 |

**Honest read:** The hand baseline Precision@50 = **0.240** sits *below* the holdout base rate (0.391) — same finding as ML-07. Random Forest lifts holdout Precision@50 to **0.680** (and Precision@20 to 0.700). Logistic is weaker but still above baseline. Numbers above are from this notebook run, not invented.


In [3]:
def build_models() -> dict[str, object]:
    return {
        "logistic_regression": Pipeline(
            steps=[
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=1000,
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
        "decision_tree": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=5,
            min_samples_leaf=50,
            random_state=RANDOM_STATE,
        ),
        "random_forest": RandomForestClassifier(
            class_weight="balanced_subsample",
            max_depth=10,
            min_samples_leaf=25,
            n_estimators=200,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        ),
    }


def predict_proba_positive(model, feature_frame: pd.DataFrame) -> np.ndarray:
    return np.asarray(model.predict_proba(feature_frame)[:, 1], dtype=float)


def ranking_metrics(y_true: pd.Series, scores: np.ndarray) -> dict[str, float]:
    return {
        "precision_at_10": precision_at_k(y_true, scores, 10),
        "precision_at_20": precision_at_k(y_true, scores, 20),
        "precision_at_50": precision_at_k(y_true, scores, 50),
        "precision_at_100": precision_at_k(y_true, scores, 100),
        "average_precision": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)),
    }


baseline_holdout = ranking_metrics(y_test, base_scores_test)

trained = build_models()
model_holdout: dict[str, dict[str, float]] = {}
model_test_scores: dict[str, np.ndarray] = {}
for name, model in trained.items():
    model.fit(X_train, y_train)
    scores = predict_proba_positive(model, X_test)
    model_test_scores[name] = scores
    model_holdout[name] = ranking_metrics(y_test, scores)

best_name = sorted(
    model_holdout,
    key=lambda n: (
        model_holdout[n]["precision_at_50"],
        model_holdout[n]["average_precision"],
        model_holdout[n]["roc_auc"],
    ),
    reverse=True,
)[0]

# comparison table
rows = [
    {
        "method": "baseline_refresh_score (ML-07)",
        "precision_at_20": baseline_holdout["precision_at_20"],
        "precision_at_50": baseline_holdout["precision_at_50"],
        "precision_at_100": baseline_holdout["precision_at_100"],
        "average_precision": baseline_holdout["average_precision"],
        "roc_auc": baseline_holdout["roc_auc"],
    }
]
for name, m in model_holdout.items():
    rows.append(
        {
            "method": name + ("  << best" if name == best_name else ""),
            "precision_at_20": m["precision_at_20"],
            "precision_at_50": m["precision_at_50"],
            "precision_at_100": m["precision_at_100"],
            "average_precision": m["average_precision"],
            "roc_auc": m["roc_auc"],
        }
    )
rows.append(
    {
        "method": "holdout base rate (chance floor)",
        "precision_at_20": base_rate_test,
        "precision_at_50": base_rate_test,
        "precision_at_100": base_rate_test,
        "average_precision": base_rate_test,
        "roc_auc": 0.5,
    }
)
compare = pd.DataFrame(rows)

print("=== client-holdout comparison (same split + metric as ML-07) ===")
print(f"holdout base rate = {base_rate_test:.3f}")
print(f"ML-07 baseline Precision@50 (holdout) = {baseline_holdout['precision_at_50']:.3f}")
print(f"best model by Precision@50 = {best_name}")
print(f"best model Precision@50 (holdout) = {model_holdout[best_name]['precision_at_50']:.3f}")
print()
print(compare.to_string(index=False, float_format=lambda x: f"{x:.3f}"))

out_dir = Path("work/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
metrics = {
    "assignment": "ML-08",
    "lane": "Refresh / Content Opportunity Scoring",
    "label": "is_declining_label = (trend_direction == 'down')",
    "split": "client_holdout",
    "random_state": RANDOM_STATE,
    "test_clients": sorted(test_clients),
    "train_rows": int(len(train_idx)),
    "test_rows": int(len(test_idx)),
    "base_rate_full": base_rate_full,
    "base_rate_test": base_rate_test,
    "feature_count": int(len(feature_columns)),
    "sklearn_version": sklearn.__version__,
    "baseline_precision_at_20_holdout": baseline_holdout["precision_at_20"],
    "baseline_precision_at_50_holdout": baseline_holdout["precision_at_50"],
    "models": model_holdout,
    "best_model": best_name,
    "best_model_precision_at_50_holdout": model_holdout[best_name]["precision_at_50"],
}
metrics_path = out_dir / "model_holdout_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
print()
print(f"Wrote metrics receipt: {metrics_path}")


=== client-holdout comparison (same split + metric as ML-07) ===
holdout base rate = 0.391
ML-07 baseline Precision@50 (holdout) = 0.240
best model by Precision@50 = random_forest
best model Precision@50 (holdout) = 0.680

                          method  precision_at_20  precision_at_50  precision_at_100  average_precision  roc_auc
  baseline_refresh_score (ML-07)            0.150            0.240             0.360              0.468    0.627
             logistic_regression            0.350            0.400             0.440              0.522    0.700
                   decision_tree            0.650            0.660             0.680              0.575    0.742
          random_forest  << best            0.700            0.680             0.700              0.610    0.747
holdout base rate (chance floor)            0.391            0.391             0.391              0.391    0.500

Wrote metrics receipt: work/outputs/model_holdout_metrics.json


## 4. Errors and interpretation

A metric without error analysis is decoration. Observed from this run (best model = **random_forest**):

**What it leans on (top importances):** `log_impressions_90d`, `days_with_impressions`, `avg_position`, `content_age_days`, then depth/CTR/scroll texture. That is plausible for a refresh queue — visibility + age + position — not a suspiciously perfect label clone. Label-source columns (`trend_direction` / `trend_pct` / `is_declining_label`) are blocked from X.

**Where it is right:** Holdout top-50 contains **34 / 50** proxy positives → Precision@50 = **0.680**; top-20 is **14 / 20** → Precision@20 = **0.700**. Far above the ML-07 baseline (0.240) and the holdout base rate (0.391).

**Where it is wrong:** Some high-score holdout pages are still proxy-stable / up — often mid-visibility pages with aged or awkward position patterns that *look* like refresh candidates. Low-score declining pages tend to be lower-visibility; the queue still prefers pages a reviewer would open first.

**Limit:** Same-window teaching proxy only — a high score means "looks like declining *in this slice*," not "refresh will recover traffic." Decision-support, not causal SEO automation.

Claims stay **observed / measured / directional / decision-support**.


In [4]:
def top_feature_importance(model, feature_columns, limit=15):
    if isinstance(model, Pipeline):
        classifier = model.named_steps["model"]
        raw = np.abs(classifier.coef_[0])
    elif hasattr(model, "feature_importances_"):
        raw = np.asarray(model.feature_importances_, dtype=float)
    else:
        raw = np.zeros(len(feature_columns))
    imp = (
        pd.DataFrame({"feature": feature_columns, "importance": raw})
        .sort_values("importance", ascending=False)
        .head(limit)
    )
    return imp


best_model = trained[best_name]
imp = top_feature_importance(best_model, feature_columns, limit=15)
print(f"=== top features ({best_name}, holdout-fitted) ===")
print(imp.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print("Sanity: top features should be plausible visibility/age/position/engagement signals.")
print("If a label-source column appeared here, that would be leakage — it does not (blocked in X).")

# holdout ranked queue for the best model
holdout_frame = df.iloc[test_idx].copy().reset_index(drop=True)
holdout_frame["model_score"] = model_test_scores[best_name]
holdout_frame["baseline_score"] = base_scores_test
holdout_ranked = holdout_frame.sort_values("model_score", ascending=False).reset_index(drop=True)
holdout_ranked["model_rank"] = np.arange(1, len(holdout_ranked) + 1)

top50 = holdout_ranked.head(50)
print()
print("=== holdout top-50 for best model ===")
print(
    f"proxy positives in top-50: {int(top50['is_declining_label'].sum())} / 50"
    f"  -> Precision@50 = {top50['is_declining_label'].mean():.3f}"
)
print(
    f"proxy positives in top-20: {int(holdout_ranked.head(20)['is_declining_label'].sum())} / 20"
    f"  -> Precision@20 = {holdout_ranked.head(20)['is_declining_label'].mean():.3f}"
)

# false positives near the top (scored high, label==0)
fp = holdout_ranked.loc[holdout_ranked["is_declining_label"] == 0].head(5)
print()
print("=== concrete wrong high-score cases (holdout; proxy-stable but ranked high) ===")
fp_cols = [
    "model_rank",
    "content_id",
    "model_score",
    "is_declining_label",
    "trend_direction",
    "impressions_90d",
    "avg_position",
    "ctr",
    "content_age_days",
    "days_since_last_update",
    "engagement_rate",
]
print(fp[fp_cols].to_string(index=False))
print()
print("Why these are hard: high visibility + aged page-1 pattern looks like refresh opportunity")
print("even when the same-window trend tag is stable — the model can still over-rank 'important' pages.")

# false negatives: declining but scored low
fn = holdout_ranked.loc[holdout_ranked["is_declining_label"] == 1].tail(5)
print()
print("=== low-score declining cases (holdout; proxy-down but ranked near bottom) ===")
print(fn[fp_cols].to_string(index=False))
print()
print("Often lower-visibility / odd-position pages: the queue still favors pages worth opening first.")

# attach top features into metrics receipt
metrics["best_model_top_features"] = [
    {"feature": str(r.feature), "importance": float(r.importance)}
    for r in imp.itertuples(index=False)
]
metrics["holdout_top50_positives"] = int(top50["is_declining_label"].sum())
metrics_path.write_text(json.dumps(metrics, indent=2) + "\n")
print()
print(f"Updated metrics receipt: {metrics_path}")


=== top features (random_forest, holdout-fitted) ===
               feature  importance
   log_impressions_90d      0.1319
 days_with_impressions      0.1304
          avg_position      0.1111
      content_age_days      0.0908
         age_tier_365+      0.0376
            word_count      0.0368
        log_clicks_90d      0.0365
            char_count      0.0363
    days_with_sessions      0.0350
                   ctr      0.0343
           scroll_rate      0.0336
days_since_last_update      0.0266
      log_sessions_90d      0.0260
   position_tier_top_3      0.0242
       age_tier_91-180      0.0195

Sanity: top features should be plausible visibility/age/position/engagement signals.
If a label-source column appeared here, that would be leakage — it does not (blocked in X).

=== holdout top-50 for best model ===
proxy positives in top-50: 34 / 50  -> Precision@50 = 0.680
proxy positives in top-20: 14 / 20  -> Precision@20 = 0.700

=== concrete wrong high-score cases (holdout; pro

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Receipt:** `work/outputs/model_holdout_metrics.json` stores the live holdout comparison (baseline vs models). Capstone notebook untouched.

*(Portal submit is out of scope for this agent run — commit + push only.)*
